# Experiment 2: COCO val2017 Training

**Purpose:** Secondary benchmark (Table 2 in manuscript). 3 seeds minimum.

**Dataset:** COCO 2017 (118K train / 5K val, 80 classes)

**Expected time:** ~6-10 hours per seed on T4 at 416×416

> ⚠️ **Disk space:** Full COCO is ~20 GB. Use `coco128.yaml` (128 images) as a lighter alternative.

In [ ]:
# === Cell 1: Setup ===
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q
!pip install tqdm -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
!df -h / | tail -1

In [ ]:
# === Cell 2: Download Dataset ===
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('coco128.yaml')"

DATA = 'coco128.yaml'  # Change to 'coco.yaml' for full COCO
print(f"\nUsing dataset: {DATA}")

In [ ]:
# === Cell 3: Configuration ===
SEEDS = [42, 123, 256]
IMGSZ = 416
EPOCHS = 300
WARMUP = 3

print(f"Plan: {len(SEEDS)} seeds × 2 variants = {len(SEEDS)*2} runs")
print(f"Dataset: {DATA} | {IMGSZ}×{IMGSZ} | {EPOCHS} epochs")

In [ ]:
# === Cell 4: Train QUANTIZED on COCO — 3 seeds ===
for i, seed in enumerate(SEEDS):
    name = f"coco-q-{IMGSZ}-seed{seed}"
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(SEEDS)}] Quantized seed={seed}")
    print(f"{'='*70}", flush=True)
    get_ipython().system(
        f"python scripts/train.py --task det --variant quantized "
        f"--data {DATA} --imgsz {IMGSZ} --epochs {EPOCHS} "
        f"--seed {seed} --warmup {WARMUP} --name {name}"
    )
print("\n🎉 Quantized COCO training complete!")

In [ ]:
# === Cell 5: Train STANDARD on COCO — 1 seed ===
print(f"{'='*70}")
print(f"  Standard seed=42")
print(f"{'='*70}", flush=True)
get_ipython().system(
    f"python scripts/train.py --task det --variant standard "
    f"--data {DATA} --imgsz {IMGSZ} --epochs {EPOCHS} "
    f"--seed 42 --warmup {WARMUP} --name coco-std-{IMGSZ}-seed42"
)
print("\n🎉 Standard COCO training complete!")

In [ ]:
# === Cell 6: Collect COCO Results ===
import json, glob, numpy as np
from pathlib import Path

print("=" * 80)
print("  COCO Results")
print("=" * 80)

coco_results = {}
for variant in ['q', 'std']:
    label = 'Quantized' if variant == 'q' else 'Standard'
    maps50, maps95 = [], []
    for f in sorted(glob.glob(f'experiments/results/coco*-{variant}-{IMGSZ}-seed*/config.json')):
        with open(f) as fh:
            cfg = json.load(fh)
        fm = cfg.get('final_metrics', {})
        m50 = fm.get('mAP50', 0)
        m95 = fm.get('mAP50_95', fm.get('mAP50-95', 0))
        maps50.append(m50); maps95.append(m95)
        print(f"  {Path(f).parent.name}: mAP@50={m50:.4f}  mAP@50-95={m95:.4f}")
    if maps50:
        print(f"  → {label}: mAP@50 = {np.mean(maps50)*100:.1f} ± {np.std(maps50)*100:.1f}%")
        print(f"             mAP@50-95 = {np.mean(maps95)*100:.1f} ± {np.std(maps95)*100:.1f}%")
        coco_results[variant] = {'mAP50': maps50, 'mAP50_95': maps95}
    print()

with open('experiments/results/coco_summary.json', 'w') as f:
    json.dump(coco_results, f, indent=2)
print("Saved: experiments/results/coco_summary.json")

In [ ]:
# === Cell 7: Download Results ===
!cd experiments && zip -r /content/coco_results.zip results/coco-*
print("📥 Download from Files panel: /content/coco_results.zip")